# Configuration

Use **pytorch** conda enviroment

# 1) Introduction

In this lesson, we'll learn how to construct neural networks using PyTorch's `nn.Sequential` [container](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html). This approach lets us stack layers together in a clean, readable way while PyTorch handles all the parameter management, initialization, and gradient tracking behind the scenes. 

We'll work with the MPG dataset from the seaborn library to build a regression model that predicts fuel efficiency (poorly).

# 2) Linear Layers and nn.Sequential

Neural networks are built from layers, and in PyTorch, the most fundamental is the `nn.Linear` layer. It performs the same "multiply then sum" operation we implemented manually in a previous lesson: **it multiplies inputs by weights, adds biases, and produces outputs**.

When we create a layer, we just specify **how many inputs** (neurons) it should expect and **how many outputs** (neurons) it should produce. The layer automatically registers its parameters with PyTorch's autograd system, so gradients will be computed and tracked without any extra work on our part. This seamless parameter management is one of the key advantages of using PyTorch's built-in layers over manual tensor operations.

The `nn.Sequential` container is what lets us stack layers and activation functions together to create a neural network. Here's an example with two Linear layers with a ReLU activation function between them:



In [1]:
import torch
import torch.nn as nn

# A simple two-layer network:
model = nn.Sequential(
    nn.Linear(in_features=3, out_features=2),
    nn.ReLU(),  # Activation function
    nn.Linear(in_features=2, out_features=1)
)
print(model)

Sequential(
  (0): Linear(in_features=3, out_features=2, bias=True)
  (1): ReLU()
  (2): Linear(in_features=2, out_features=1, bias=True)
)


Notice how the layers connect: the first layer outputs 2 values (`out_features=2`), and the second layer expects exactly 2 inputs (`in_features=2`). This isn't coincidence, it's required! Each layer's output size must match the next layer's input size. 

Mismatched dimensions (like outputting 5 values when the next layer expects 8) will cause PyTorch to throw an error. When data passes through our model, each layer transforms it sequentially before passing the result to the next layer.

Similarly, our input data must match the expected input size of the first (input) layer. Since the input layer of our model expects 3 features (`in_features=3`), our input data for this model must have 3 features.

In [2]:
# Create some sample input data
input_data = torch.randn(4, 3)

# Forward pass - data flows through all layers
output = model(input_data)

print(f"Output shape: {output.shape}\n")
print(output)

Output shape: torch.Size([4, 1])

tensor([[0.1775],
        [0.3783],
        [0.1847],
        [0.0732]], grad_fn=<AddmmBackward0>)


In `torch.randn(4, 3)`, the shape `[4, 3]` represents 4 samples with 3 features each. The model processes all samples simultaneously, but each sample must have exactly 3 features. The model doesn't care how many samples we provide (more is better!), but the feature count must match what the first layer expects as input (`3`).

Also, notice how we just call `model(input_data)` to generate model predictions (`output`). There's no need for us to manually compute each layer's output because PyTorch handles the forward propagation automatically, applying each layer in the order we specified when creating the model.

The `output` above shows us a couple of important things:

1. The shape `[4, 1]` confirms our model processed all 4 input samples and produced 1 prediction value for each, exactly what we'd expect for a regression model.

1. `grad_fn` tells us that PyTorch is automatically tracking how this tensor was created so it can compute gradients later during training. This is autograd working behind the scenes. We'll explore how PyTorch uses these gradients for training in our next lesson.

## Instructions

1. Create a neural network called `simple_model` using nn.Sequential with:

* First layer: `nn.Linear` layer that takes 8 inputs and produces 4 outputs

* Activation: `nn.ReLU()` function

* Second layer: `nn.Linear` layer that takes 4 inputs and produces 1 output

1. Print `simple_model` to see its structure.

1. Use the `torch.randn` function to create an input tensor test_input with 10 random samples and the number of features `simple_model` expects as input.

1. Pass `test_input` through your model and store the output in test_output.

1. Print the shape of `test_output` to confirm it has the expected shape for 10 inputs with 1 output.

In [3]:
import torch
import torch.nn as nn

simple_model = nn.Sequential(
    nn.Linear(in_features=8,out_features=4),
    nn.ReLU(),
    nn.Linear(in_features=4,out_features=1)
)

print(simple_model)

test_input = torch.randn(10,8)
test_output = simple_model(test_input)

print(test_output)

Sequential(
  (0): Linear(in_features=8, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)
tensor([[0.3093],
        [0.1394],
        [0.2665],
        [0.3168],
        [0.2238],
        [0.2493],
        [0.0997],
        [0.2650],
        [0.1686],
        [0.1636]], grad_fn=<AddmmBackward0>)


# 3) Loading and Exploring the MPG Dataset

let's see what it takes to prepare a dataset to feed into it. We'll use the MPG (miles per gallon) dataset, which contains information about various car models and their fuel efficiency.

To load this dataset, we'll use seaborn's `load_dataset()` function with the dataset name 'mpg': `sns.load_dataset('mpg')`. This function provides easy access to built-in datasets perfect for learning and experimentation. To get a full list of built-in datasets, use `sns.get_dataset_names()`.

The MPG dataset contains the following columns:

**mpg**: Measures fuel efficiency in miles per gallon (this is our target variable to predict)

**cylinders**: Number of engine cylinders

**displacement**: Engine displacement in cubic inches

**horsepower**: Engine horsepower

**weight**: Vehicle weight in pounds

**acceleration**: Time to accelerate from 0-60 mph in seconds

**model_year**: Model year (70s and 80s cars)

**origin**: Country of origin (usa, europe, japan)

**name**: Car model name (not useful for prediction, we'll remove this column)

When we examine datasets for machine learning, we're looking for several key things:

**Missing values** that need to be handled

**Categorical variables** that need conversion to numerical values

**Which column** serves as our target variable

To explore our dataset, we'll use the following pandas methods:

* `info()` method to see column types and missing values

* `head()` method to view sample rows

* `isnull()` method combined with the `sum()` method to count missing values per column

In the exercise below, you'll load the MPG dataset and perform the exploration steps we need to understand our data before feeding it into our model.

## Instructions

1. Load the MPG dataset and store it in `df`.

1. Display information about the dataset to determine if there are any missing values and take a look at the first 5 rows.

1. Check for missing values across all columns and identify which column has missing values and how many. Store the column name as a string, and the count of missing values as an integer in a tuple called `missing_info`.

1. Excluding the 'name' column, identify which column contains categorical data that will need conversion to numerical values. Store this column name as a string in `categorical_column`.

In [4]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset('mpg')

print(df.head())
missing_values = df.isnull().sum()
missing_column = missing_values[missing_values > 0].index[0]
missing_count = 6
missing_info = (missing_column, missing_count)
categorical_column = 'origin'

    mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   
3  16.0          8         304.0       150.0    3433          12.0   
4  17.0          8         302.0       140.0    3449          10.5   

   model_year origin                       name  
0          70    usa  chevrolet chevelle malibu  
1          70    usa          buick skylark 320  
2          70    usa         plymouth satellite  
3          70    usa              amc rebel sst  
4          70    usa                ford torino  


# 4) Handling Categorical Data with Dummy Encoding

From our exploration of the data, we saw that the **'origin'** column contains categorical values representing 3 different countries. Neural networks can only work with numbers, so we need to convert these categories into a **numeric** format.

The standard approach is to implement dummy encoding using pandas' `get_dummies()` function. This technique creates binary columns where each category gets its own column, with 1 indicating that category, and 0 indicating it doesn't belong to that category.

Before conversion, the 'origin' column contains only these text values: 'usa', 'europe', or 'japan'. 

# 5) Creating Training Data and Normalizing Features

With our dataset cleaned and encoded, our next step is to prepare it for our neural network. This involves three important steps:

1. Separating features from targets

1. Normalizing the features

1. Converting everything to PyTorch tensors

**Why normalize features?**

Neural networks train more effectively when input features are on similar scales. Without normalization, features with large values (like 'weight' in pounds) would dominate features with small values (like 'acceleration' in seconds). We'll use standardization, which transforms each feature to have **zero mean** and **unit variance**

Here's the complete data preparation workflow:

```python
# Separate features and target
X = df_name.drop('target_variable', axis=1).values  # Returns features as NumPy array
y = df_name['target_variable'].values               # Returns target as NumPy array

# Normalize features (calculate statistics column-wise)
X_normalized = (X - X.mean(axis=0)) / X.std(axis=0)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_normalized, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
```

Notice that we reshape y to have shape `(-1, 1)` instead of just `(-1,)`. This ensures our target has the right shape for future calculations: each sample has one output value, but we want it as a column vector rather than a 1D array. 

The `-1` is a special value that tells PyTorch to automatically calculate that dimension's size based on the **total number of elements**. For example, if we have N samples, `(-1, 1)` becomes `(N, 1)` and `(1, -1)` would become `(1, N)`.

Specifying `dtype=torch.float32` is important because neural networks typically work best with 32-bit floats. It's more memory-efficient than 64-bit floats while still providing enough precision for training.

## Instructions

1. Create the features array `X` by dropping the `'mpg'` column from `df` and converting to a NumPy array.

1. Create the target array `y` by extracting the 'mpg' column and converting to a NumPy array.

1. Normalize the features array `X` and store in `X_normalized`.

1. Convert the normalized feature array to a PyTorch tensor called `X_tensor` with float32 values.

1. Convert the target array `y` to a PyTorch tensor called `y_tensor` with float32 values and reshape it to `(-1, 1)`.

1. Print the shapes of both tensors (`X_tensor` and `y_tensor`) and the first 3 values of each to confirm your results.

In [6]:
import torch

df = sns.load_dataset('mpg')
df = df.drop('name', axis=1)
df = df.dropna()
df = pd.get_dummies(df, columns=['origin'], drop_first=True, dtype=float)

X = df.drop('mpg',axis=1).values
y = df['mpg'].values

#Normalize features
X_normalized = (X - X.mean(axis=0)) / X.std(axis=0)

#Convert to Pytorch tensors
X_tensor = torch.tensor(X_normalized, dtype= torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1,1) #column vector

print(X_tensor[:3],y_tensor[0:3])

tensor([[ 1.4839,  1.0773,  0.6641,  0.6205, -1.2853, -1.6253, -0.5024,  0.7746],
        [ 1.4839,  1.4887,  1.5746,  0.8433, -1.4667, -1.6253, -0.5024,  0.7746],
        [ 1.4839,  1.1825,  1.1844,  0.5404, -1.6482, -1.6253, -0.5024,  0.7746]]) tensor([[18.],
        [15.],
        [18.]])


# 6) Building a Regression Model for MPG

From the data preparation in the previous screen, we know our input has 8 features (after dropping 'mpg' and 'name', and encoding 'origin'), and we want to predict a single continuous value (MPG). This means our model will **take in 8 features**, and produce a **single output** for each sample we feed into it.

When designing a neural network architecture, we need to decide:

* How many hidden layers to use

* How many neurons to use in each layer

* Which activation functions to apply

For regression tasks like predicting MPG, here's a typical architecture we could use:

```python
model = nn.Sequential(
    nn.Linear(input_features, 32),  # First hidden layer
    nn.ReLU(),
    nn.Linear(32, 16),              # Second hidden layer  
    nn.ReLU(),
    nn.Linear(16, output_features)  # Output layer (no activation)
)
```

Notice that the **output layer doesn't have an activation function**. For regression, we want the network to output any real value, not restrict it to a specific range. If we used an activation function on the output layer, we'd limit what MPG values the model could predict.

The choice of hidden layer sizes (32 or 16 neurons) follows a common pattern: start with more neurons and gradually decrease. This allows the network to learn complex patterns in early layers and combine them into the final prediction. However, these aren't strict rules. Network architecture often requires experimentation, and it's often more of an art than a science.

You can access the number of input features programmatically from the features tensor:

```python
num_input_features = X_tensor.shape[1]  # Gets the number of columns/features
```

This is considered best practice since it allows us to dynamically set the `in_features` of the first layer to avoid sizing errors.

## Instructions

1. Get the number of input features programmatically and store it in input_size.

1. Build an mpg_model using nn.Sequential with:

    * First layer: nn.Linear from input_size to 64 neurons

    * ReLU activation

    * Second layer: nn.Linear from 64 to 32 neurons

    * ReLU activation

    * Output layer: nn.Linear from 32 to 1 neuron

1. Print the model architecture.

1. Pass X_tensor through mpg_model and store the results in initial_predictions.

1. Print the shape of initial_predictions and y_tensor to verify they match.


In [7]:
import torch.nn as nn

X = df.drop('mpg', axis=1).values
y = df['mpg'].values
X_normalized = (X - X.mean(axis=0)) / X.std(axis=0)
X_tensor = torch.tensor(X_normalized, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

input_size = X_tensor.shape[1]

mpg_model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,1)
)

print(mpg_model)

initial_predictions = mpg_model(X_tensor)

print(initial_predictions,y_tensor)
    
    
    

Sequential(
  (0): Linear(in_features=8, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)
tensor([[-0.1754],
        [-0.1900],
        [-0.1940],
        [-0.1841],
        [-0.1953],
        [-0.1961],
        [-0.2138],
        [-0.2188],
        [-0.2060],
        [-0.2250],
        [-0.2068],
        [-0.2224],
        [-0.1903],
        [-0.2094],
        [-0.3047],
        [-0.1790],
        [-0.1802],
        [-0.1942],
        [-0.3159],
        [-0.3148],
        [-0.2806],
        [-0.2473],
        [-0.2909],
        [-0.2665],
        [-0.1840],
        [-0.1971],
        [-0.1859],
        [-0.1963],
        [-0.1767],
        [-0.3023],
        [-0.2021],
        [-0.2857],
        [-0.1570],
        [-0.1542],
        [-0.1489],
        [-0.1487],
        [-0.1532],
        [-0.1910],
        [-0.1840],
        [-0.1801],
        [-0.1796],
       

# 7) Working with Different Activation Functions

So far we've used ReLU activation functions between our layers. Without activation functions, neural networks would just be a series of linear transformations, and no matter how many layers we stack, the result would still be equivalent to a single linear layer. **Activation functions introduce a nonlinearity factor that allows networks to learn complex patterns.**

PyTorch provides many activation functions through the torch.nn module. Each has different properties that make them suitable for different situations.

## **Common Activation Functions**

**For Output Layers:**

* `nn.Sigmoid():` Squashes values to range of (0, 1). Perfect for binary classification since outputs can be interpreted as probabilities.

**For Hidden Layers:**

* `nn.ReLU():` Outputs `max(0, x)`. Simple, efficient, and the most popular choice for hidden layers. Helps avoid the vanishing gradient problem (where gradients become too small to effectively update weights in deep networks).

* `nn.Tanh()`: Squashes values to (-1, 1). Zero-centered output can help with convergence (how quickly the network reaches optimal weights during training).

* `nn.LeakyReLU()`: Like ReLU but allows small negative values to pass through. Prevents "dead neurons" (neurons that get stuck outputting zero and stop learning because ReLU kills all negative inputs).

The `negative_slope` parameter in `LeakyReLU` controls how much of the negative input passes through. A slope of `0.1` means negative inputs are reduced to 10% of their original value rather than being completely zeroed out.

Here's how different activations transform the same input values:

In [8]:
import torch
import torch.nn as nn

# Test with a range of values: negative, zero, and positive
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

# All outputs between 0 and 1
sigmoid = nn.Sigmoid()
print(f"Sigmoid: {sigmoid(x)}")  # Sigmoid: tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])

# Negative values become 0, otherwise unchanged
relu = nn.ReLU()
print(f"ReLU: {relu(x)}")  # ReLU: tensor([0., 0., 0., 1., 2.])

# All outputs between -1 and 1, centered at 0
tanh = nn.Tanh()
print(f"Tanh: {tanh(x)}")  # Tanh: tensor([-0.9640, -0.7616, 0.0000, 0.7616, 0.9640])

# Negative values become small but non-zero (multiplied by 0.1)
leaky = nn.LeakyReLU(negative_slope=0.1)
print(f"LeakyReLU: {leaky(x)}")  # LeakyReLU: tensor([-0.2000, -0.1000, 0.0000, 1.0000, 2.0000])

Sigmoid: tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
ReLU: tensor([0., 0., 0., 1., 2.])
Tanh: tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])
LeakyReLU: tensor([-0.2000, -0.1000,  0.0000,  1.0000,  2.0000])


## Choosing the Right Activation

**General guidelines:**

* **Hidden layers:** Start with ReLU. It's simple, fast, and works well in most cases

* **Binary classification output**: Use Sigmoid to get probabilities between 0 and 1

* **Regression output**: No activation (let the linear layer output any real number)

* **If ReLU causes training problems**: Try LeakyReLU or Tanh

The choice of activation function can significantly impact training speed and final performance, but ReLU is an excellent default for most hidden layers.

Now it's time for you to build an alternative version of our MPG model using different activation functions to see how they affect the network's behavior.

## Instructions

1. Create alt_model using nn.Sequential with:

    * **First layer**: nn.Linear from input_size to 64

    * nn.Tanh() activation

    * **Second layer**: nn.Linear from 64 to 32

    * nn.LeakyReLU() activation with negative values reduced to 5% of their original value

    * **Output layer**: nn.Linear from 32 to 1

1. Make predictions with `alt_model` on the first 5 samples of `X_tensor`. Store your results in `sample_predictions` as a 1D tensor by calling view() with the appropriate argument.

1. Make predictions with the original mpg_model on the same samples. Store your results in original_predictions as a 1D tensor by calling view() with the appropriate argument.

1. Use print() to display both sets of predictions to see how different activations affect the output.

1. Print the first five values of `y_tensor` as a 1D tensor to compare the actual values to your models' predictions above.

In [9]:
import torch.nn as nn

input_size = X_tensor.shape[1]
mpg_model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

alt_model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.Tanh(),
    nn.Linear(64,32),
    nn.LeakyReLU(negative_slope=0.05),
    nn.Linear(32,1)
)

sample_predictions = alt_model(X_tensor[:5]).view(-1) #first 5 predictions. view with 1D dimension
original_predictions = mpg_model(X_tensor[:5]).view(-1)

print(sample_predictions, original_predictions)
print(y_tensor[:5].view(-1))


tensor([0.4018, 0.3894, 0.4257, 0.4068, 0.4450], grad_fn=<ViewBackward0>) tensor([-0.2713, -0.2672, -0.2669, -0.2664, -0.2617], grad_fn=<ViewBackward0>)
tensor([18., 15., 18., 16., 17.])


# 8) Understanding Model Parameters

In the previous exercise, you compared predictions from two models with different activation functions.  Notice that both models produced predictions that were **very different from the actual MPG values** they were meant to predict, which is completely expected! Since we haven't trained our models yet, their randomly initialized parameters produced random predictions.

The purpose of that exercise was to demonstrate how different activation functions affect data flow through the network, not to make accurate predictions. Training, which we'll cover in the next lesson, is what allows our model to make useful predictions by adjusting model parameters to better predict our target variable.

So what exactly are these parameters that training will adjust? Every neural network contains **parameters** (**weights and biases**) that determine how the network transforms input data into predictions. With `nn.Sequential`, PyTorch automatically creates and manages these parameters for us.

Each nn.Linear layer in our model contains two types of parameters:

* **Weights**: A matrix that transforms inputs to outputs through multiplication

* **Bias**: A vector added to each output to shift values

## Accessing Individual Layers

Since `nn.Sequential` acts like a list, we can access layers by indexing the model object:

```python
layer_1 = model[0]    # Gets the first layer (index 0)
layer_2 = model[1]    # Gets the second layer (index 1)
```

Each layer's parameters are stored as `.weight` and `.bias` attributes. We can also access their `.shape` attributes to confirm their dimensions:

```python
print(layer_1.weight.shape)   # Shows weight matrix dimensions
print(layer_2.bias.shape)     # Shows bias vector dimensions
```

## Understanding Parameter Shapes

Parameter shapes follow a specific pattern that can seem backwards at first. Most people intuitively think of matrix multiplication as 

$input × weight = output$

which would suggest a weight matrix shaped like (`in_features`, `out_features`). However, PyTorch (like most deep learning frameworks) uses the opposite convention:

* A `nn.Linear(8, 64)` layer has weights shaped `(64, 8)` and bias shaped `(64)`

* The weight matrix uses (`out_features`, `in_features`) ordering (outputs first, inputs second)

* The bias always matches the number of outputs

This happens because of how PyTorch performs the underlying matrix multiplication. The weight matrix gets transposed (its dimensions are reversed) during computation for efficiency, so it's stored in the transposed form.

## Examining All Parameters

The `named_parameters()` method lets us loop through every parameter in our model, giving us both the parameter's name and the tensor containing its values. This is useful for getting a complete overview of our model's structure:

```python
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
```

This code will output something like:

```python
0.weight: torch.Size([64, 8])
0.bias: torch.Size([64])
2.weight: torch.Size([32, 64])
2.bias: torch.Size([32])
4.weight: torch.Size([1, 32])
4.bias: torch.Size([1])
```

Parameter names follow the pattern `layer_index`.`parameter_type`. Notice how activation functions (like ReLU) don't appear in the list. Since they don't have parameters to update, layers 1 and 3 (the activations) are skipped in the numbering.

## Counting Parameters

The `numel()` method returns the total number of elements in a tensor. To count all parameters in a model, we can combine it with the `parameters()` method, which returns just the parameter tensors without the names. This is perfect for when we want to understand the complexity of our model:

```python
total_params = sum(param.numel() for param in model.parameters())
```

While models with more parameters can learn more complex patterns, they require more computational resources and training data to avoid overfitting. This happens when the model learns the training data too specifically rather than learning general patterns.

All parameters are automatically initialized as `float32` tensors with small random values when layers are first created. PyTorch uses research-backed initialization techniques designed to help networks train more effectively.

## Instructions

1. Access the first layer of `mpg_model` using indexing and store it in `first_layer`.

1. Get the `.shape` of the `.weight` and `.bias` attributes of `first_layer` and store them in `first_layer_weight_shape` and `first_layer_bias_shape`, respectively.

1. Loop through the model's named parameters and print each parameter's name and shape.

1. Calculate the total number of parameters in your mpg_model. Store your results in `total_params`.

1. Use print() to display `first_layer_weight_shape`, `first_layer_bias_shape`, and `total_params` to confirm your results.

In [12]:
import torch.nn as nn

input_size = X_tensor.shape[1]
mpg_model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

first_layer = mpg_model[0]
first_layer_weight_shape = first_layer.weight.shape
first_layer_bias_shape = first_layer.bias.shape

for name, param in mpg_model.named_parameters():
    print(f"{name}: {param.shape}")
          
total_params = sum(param.numel() for param in mpg_model.parameters())

print(first_layer_weight_shape, first_layer_bias_shape, total_params)
          
    
          
          

0.weight: torch.Size([64, 8])
0.bias: torch.Size([64])
2.weight: torch.Size([32, 64])
2.bias: torch.Size([32])
4.weight: torch.Size([1, 32])
4.bias: torch.Size([1])
torch.Size([64, 8]) torch.Size([64]) 2689


# Train-Test Split for Model Evaluation

Before we can properly evaluate our model's performance, we need to split our data into **training** and **testing** sets. By training the model on one portion of our data and then evaluating it on a completely separate portion it has never seen, we get an honest assessment of how well the model will perform on new, unseen data.

This fundamental ML practice prevents overfitting and tells us whether our model has learned generalizable patterns or simply memorized the training examples.

The `train_test_split` function from scikit-learn handles this splitting while maintaining the correspondence between features and targets:

```python
from sklearn.model_selection import train_test_split

# Split data: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.3, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
```

**Key parameters:**

* `test_size`: Fraction of data to reserve for testing (0.3 = 30%)

* `random_state`: Ensures reproducible splits across code runs

After splitting, we'll have four tensors:

* `X_train`, `y_train`: For training the model (X = features, y = targets)

* `X_test`, `y_test`: For evaluating performance on unseen data

In practice, you might also create a validation set (a third split) for tuning hyperparameters, but for now we'll stick with the simpler train-test split.

In [13]:
import torch.nn as nn

input_size = X_tensor.shape[1]
mpg_model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)


from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

# Print training and testing shapes
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Make training predictions
train_predictions = mpg_model(X_train)

# Confirm predictions and actual values have the same shape
print(train_predictions.shape)
print(y_train.shape)

torch.Size([313, 8])
torch.Size([79, 8])
torch.Size([313, 1])
torch.Size([79, 1])
torch.Size([313, 1])
torch.Size([313, 1])


# 10) Computing Loss Without Training

While we won't train our model in this lesson, we can already compute the loss to see how well our randomly initialized model was able to predict the expected MPG values. This gives us a baseline to compare against once we do start training our model.

For regression tasks, **Mean Squared Error (MSE)** is the standard loss function. PyTorch provides the `nn.MSELoss()` class which computes:

$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$

Where $y_i$ is the true value (from our dataset) and $\hat{y}_i$ is the predicted value (from our model's output).

```python
# Create the loss function
criterion = nn.MSELoss()

# Make predictions and compute loss (no gradients needed during evaluation)
with torch.no_grad():
    predictions = model(data)
    loss = criterion(predictions, y_train)

print(loss.item())
```

We use the `torch.no_grad()` context manager when evaluating models since we don't need gradient computation for loss calculation, which saves memory and improves performance. Recall that the `item()` method extracts a scalar value from a single-element tensor, which is useful for printing and logging.

A few important notes about MSE:

* MSE calculates the mean of squared differences between predictions and actual values. Because differences are squared, larger errors are penalized much more heavily than smaller ones.

* MSE values are in squared units compared to our original target variable. Since we're predicting MPG and squaring the errors, our MSE is in "squared MPG" units, which makes direct comparison to actual MPG values difficult.

* To get a loss metric on the same scale as our targets, we'd need to take the square root of MSE (called Root Mean Square Error or RMSE). An RMSE of 5.0 would mean predictions are off by about 5 MPG on average.

## Instructions

1. Create an MSE loss function using `nn.MSELoss()` and store it in criterion.

1. Make predictions on X_train using mpg_model and store your results in train_preds. Use torch.no_grad() context manager for all evaluation steps since we don't need gradients.

1. Compute the training loss between train_preds and y_train. Store your results in train_loss.

1. Make predictions on X_test and compute the loss for the test data. Store your results in test_preds and test_loss, respectively.

1. Print both loss values using the item() method to extract the scalar values.


In [14]:
import torch.nn as nn
from sklearn.model_selection import train_test_split

input_size = X_tensor.shape[1]
mpg_model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

#Create the loss function
criterion = nn.MSELoss()

#Make predictions

with torch.no_grad():
        train_preds = mpg_model(X_train)
        train_loss = criterion(train_preds, y_train)
        
        test_preds = mpg_model(X_test)
        test_loss = criterion(test_preds, y_test)
        
print(train_loss.item(), test_loss.item())

626.9151611328125 579.4237670898438
